# CartilAI — Kaggle training notebook
**AI-assisted knee MRI analysis.** Research prototype — not for clinical use.

This notebook trains the CartilAI model on the **MRNet** knee MRI dataset and saves everything the app needs.
It is self-contained: attach an MRNet dataset, turn on a **GPU (T4)** and **Internet**, then click **Run All**. No edits needed.

**What it does**
1. Finds the MRNet data under `/kaggle/input` automatically (any mirror with the standard MRNet layout).
2. Trains a ResNet18 (ImageNet-pretrained) applied to every sagittal slice, max-pooled across slices, with one output per finding: *Abnormal*, *ACL tear*, *Meniscal tear* (the MRNet-paper approach).
3. Picks the best epoch and decision thresholds on an internal validation split carved out of MRNet *train*.
4. Evaluates once on the official MRNet *validation* split (never seen during training): AUC with 95% CIs, sensitivity, specificity, ROC curves, confusion matrices.
5. Exports the weights, the results and a few sample scans to `/kaggle/working/cartilai_outputs/`, zipped as **`cartilai_outputs.zip`**.

Training stops by itself before a fixed time budget, so the whole run finishes in well under 2 hours on a T4.

_Generated from the CartilAI repository by `scripts/build_notebook.py`; do not edit by hand._

In [ ]:
# ===== Setup: environment, paths and configuration =====
import os, sys, time, json, shutil, zipfile, warnings
from pathlib import Path
import numpy as np
import torch
warnings.filterwarnings("ignore", category=UserWarning)

INPUT_DIR = Path(os.environ.get("CARTILAI_INPUT_DIR", "/kaggle/input"))
WORK_DIR = Path(os.environ.get("CARTILAI_OUTPUT_DIR", "/kaggle/working"))
OUTPUT_DIR = WORK_DIR / "cartilai_outputs"
QUICK = os.environ.get("CARTILAI_QUICK") == "1"   # tiny smoke-test settings (used by the repo's tests)

CONFIG = json.loads(r"""{
  "seed": 42,
  "findings": [
    "abnormal",
    "acl",
    "meniscus"
  ],
  "finding_names": {
    "abnormal": "Abnormal",
    "acl": "ACL tear",
    "meniscus": "Meniscal tear"
  },
  "data": {
    "plane": "sagittal",
    "image_size": 256,
    "internal_val_fraction": 0.1,
    "num_workers": 2
  },
  "model": {
    "backbone": "resnet18",
    "pretrained": true,
    "dropout": 0.3
  },
  "train": {
    "epochs": 16,
    "lr": 3e-05,
    "weight_decay": 0.0001,
    "max_train_minutes": 85,
    "augment": true,
    "rotation_degrees": 25,
    "translate_pixels": 25,
    "hflip": true,
    "amp": true
  },
  "paths": {
    "models_dir": "models",
    "results_dir": "results",
    "samples_dir": "samples",
    "weights_file": "models/cartilai_resnet18.pt",
    "metrics_file": "results/metrics.json"
  },
  "report": {
    "likely_threshold_margin": 0.15
  }
}""")
if QUICK:
    CONFIG["data"].update({"image_size": 96, "num_workers": 0, "internal_val_fraction": 0.25})
    CONFIG["model"]["pretrained"] = False
    CONFIG["train"].update({"epochs": 2, "lr": 1e-4, "max_train_minutes": 10})

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print(f"GPU: {name}")
    try:  # some PyTorch builds no longer support older GPUs such as the P100
        (torch.ones(8, device="cuda") * 2).sum().item()
    except Exception as exc:
        raise RuntimeError(
            f"The GPU '{name}' does not work with this PyTorch version ({exc}). "
            "In the right-hand panel choose Settings > Accelerator > 'GPU T4 x2' and Run All again."
        )
elif not QUICK:
    raise RuntimeError(
        "No GPU found. In the right-hand panel open 'Session options' (or Settings) > Accelerator, "
        "choose 'GPU T4 x2', then click Run All again. Training on CPU would take many hours."
    )
print("Input folder:", INPUT_DIR)
print("Outputs will be written to:", OUTPUT_DIR)

## Pipeline code
The next cells contain the CartilAI source code (data loading, model, evaluation, training). They only define functions.

In [ ]:
# ===== src/config.py (inlined) =====
"""Configuration loading and reproducibility helpers."""

from __future__ import annotations

import copy
import os
import random
from pathlib import Path
from typing import Any

import numpy as np
import torch

PROJECT_ROOT = Path(".")
DEFAULT_CONFIG_PATH = Path("config.yaml")


def load_config(path: str | Path | None = None) -> dict[str, Any]:
    """Load the YAML config file into a plain dictionary.

    Args:
        path: Optional path to a YAML file. Defaults to ``config.yaml`` at the repo root.

    Returns:
        The configuration as nested dictionaries.
    """
    import yaml

    with open(path or DEFAULT_CONFIG_PATH, "r", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def merge_config(base: dict[str, Any], overrides: dict[str, Any]) -> dict[str, Any]:
    """Recursively merge ``overrides`` into a copy of ``base``.

    Args:
        base: The base configuration.
        overrides: Values that replace or extend ``base``.

    Returns:
        A new merged dictionary; inputs are not modified.
    """
    out = copy.deepcopy(base)
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = merge_config(out[key], value)
        else:
            out[key] = copy.deepcopy(value)
    return out


def set_seed(seed: int) -> None:
    """Fix all random seeds (Python, NumPy, PyTorch) for reproducible runs.

    Args:
        seed: The seed value.
    """
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False  # keep cuDNN fast; seeds still fix init and data order
    torch.backends.cudnn.benchmark = True


In [ ]:
# ===== src/data.py (inlined) =====
"""MRNet data loading, preprocessing and augmentation.

MRNet layout (Stanford ML Group, v1.0)::

    MRNet-v1.0/
        train/{axial,coronal,sagittal}/0000.npy ... 1129.npy
        valid/{axial,coronal,sagittal}/1130.npy ... 1249.npy
        train-abnormal.csv  train-acl.csv  train-meniscus.csv
        valid-abnormal.csv  valid-acl.csv  valid-meniscus.csv

Each ``.npy`` is a uint8 volume of shape ``(slices, 256, 256)``; each CSV has
rows ``case_id,label`` without a header.
"""

from __future__ import annotations

import csv
import math
import os
from pathlib import Path
from typing import Sequence

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
MAX_SLICES = 128


def find_mrnet_root(search_dir: str | Path, plane: str = "sagittal") -> Path:
    """Locate the MRNet root folder anywhere below ``search_dir``.

    Kaggle mirrors nest the data differently (``/kaggle/input/<name>/MRNet-v1.0``,
    ``/kaggle/input/<name>/data``...). We look for the folder that contains
    ``train-acl.csv`` and a ``train/<plane>`` sub-folder.

    Args:
        search_dir: Directory to search recursively.
        plane: MRNet plane that must be present.

    Returns:
        Path to the MRNet root.

    Raises:
        FileNotFoundError: If no MRNet-shaped folder exists below ``search_dir``.
    """
    search_dir = Path(search_dir)
    for dirpath, dirnames, filenames in os.walk(search_dir):
        lower = {f.lower() for f in filenames}
        if "train-acl.csv" in lower and (Path(dirpath) / "train" / plane).is_dir():
            return Path(dirpath)
        dirnames[:] = [d for d in dirnames if d not in {"axial", "coronal", "sagittal"}]
    raise FileNotFoundError(
        f"Could not find an MRNet dataset (train-acl.csv + train/{plane}/) under {search_dir}. "
        "Attach an MRNet dataset to the notebook (see KAGGLE_STEPS.md)."
    )


def _find_csv(root: Path, name: str) -> Path:
    """Return the CSV ``name`` inside ``root`` regardless of filename case."""
    for path in root.iterdir():
        if path.name.lower() == name.lower():
            return path
    raise FileNotFoundError(f"Missing label file {name} in {root}")


def read_labels(root: str | Path, split: str, findings: Sequence[str]) -> tuple[list[str], np.ndarray]:
    """Read MRNet label CSVs for a split.

    Args:
        root: MRNet root folder.
        split: ``"train"`` or ``"valid"``.
        findings: Finding keys, e.g. ``["abnormal", "acl", "meniscus"]``.

    Returns:
        ``(case_ids, labels)`` where ``labels`` has shape ``(n_cases, n_findings)``.
    """
    root = Path(root)
    per_finding: list[dict[str, int]] = []
    for finding in findings:
        mapping: dict[str, int] = {}
        with open(_find_csv(root, f"{split}-{finding}.csv"), newline="", encoding="utf-8") as fh:
            for row in csv.reader(fh):
                if len(row) < 2 or not row[0].strip().isdigit():
                    continue  # skip an optional header line
                mapping[row[0].strip().zfill(4)] = int(float(row[1]))
        per_finding.append(mapping)
    case_ids = sorted(set.intersection(*(set(m) for m in per_finding)))
    labels = np.array([[m[c] for m in per_finding] for c in case_ids], dtype=np.float32)
    return case_ids, labels


def to_uint8_volume(volume: np.ndarray, image_size: int = 256) -> np.ndarray:
    """Validate and normalise an arbitrary 3D array into a ``(S, H, W)`` uint8 volume.

    Accepts MRNet-style ``(S, H, W)`` arrays, squeezes singleton dimensions,
    rescales non-uint8 intensities to 0..255 and resizes slices to
    ``image_size`` if needed.

    Args:
        volume: Input array.
        image_size: Target in-plane size.

    Returns:
        A uint8 array of shape ``(S, image_size, image_size)``.

    Raises:
        ValueError: If the array cannot be interpreted as a 3D volume.
    """
    arr = np.asarray(volume)
    arr = np.squeeze(arr)
    if arr.ndim == 2:
        arr = arr[None]
    if arr.ndim != 3:
        raise ValueError(f"Expected a 3D volume (slices, height, width); got shape {np.shape(volume)}")
    # MRNet stores slices first. If the smallest axis is last, move it to the front.
    if arr.shape[2] < min(arr.shape[0], arr.shape[1]) and arr.shape[0] == arr.shape[1]:
        arr = np.moveaxis(arr, 2, 0)
    if arr.shape[0] > MAX_SLICES:
        raise ValueError(f"Volume has {arr.shape[0]} slices; at most {MAX_SLICES} are supported")
    if not np.isfinite(arr).all():
        arr = np.nan_to_num(arr.astype(np.float32))
    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        lo, hi = float(arr.min()), float(arr.max())
        arr = ((arr - lo) / (hi - lo) * 255.0) if hi > lo else np.zeros_like(arr)
        arr = arr.round().astype(np.uint8)
    if arr.shape[1] != image_size or arr.shape[2] != image_size:
        t = torch.from_numpy(arr).float()[:, None]
        t = F.interpolate(t, size=(image_size, image_size), mode="bilinear", align_corners=False)
        arr = t[:, 0].clamp(0, 255).round().to(torch.uint8).numpy()
    return np.ascontiguousarray(arr)


def preprocess(volume: torch.Tensor) -> torch.Tensor:
    """Turn a uint8 volume ``(S, H, W)`` into a normalised network input ``(S, 3, H, W)``.

    Slices are scaled to [0, 1], replicated to 3 channels and normalised with
    ImageNet statistics so they match what the pretrained backbone expects.

    Args:
        volume: uint8 (or float 0..255) tensor of shape ``(S, H, W)``.

    Returns:
        Float tensor of shape ``(S, 3, H, W)`` on the same device.
    """
    x = volume.float().div(255.0).unsqueeze(1).expand(-1, 3, -1, -1)
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std


def augment(
    x: torch.Tensor,
    generator: torch.Generator | None = None,
    rotation_degrees: float = 25.0,
    translate_pixels: float = 25.0,
    hflip: bool = True,
) -> torch.Tensor:
    """Apply one random affine transform (and optional flip) to every slice of a volume.

    Mirrors the MRNet paper augmentation: random rotation, translation and
    horizontal flip, shared by all slices of the exam.

    Args:
        x: Tensor ``(S, C, H, W)``.
        generator: Optional CPU random generator for reproducibility.
        rotation_degrees: Max absolute rotation.
        translate_pixels: Max absolute shift in pixels.
        hflip: Whether to randomly flip horizontally.

    Returns:
        Augmented tensor with the same shape.
    """
    s, _, h, w = x.shape
    r = torch.rand(4, generator=generator)
    angle = math.radians((r[0].item() * 2 - 1) * rotation_degrees)
    tx = (r[1].item() * 2 - 1) * translate_pixels * 2 / w
    ty = (r[2].item() * 2 - 1) * translate_pixels * 2 / h
    flip = -1.0 if (hflip and r[3].item() < 0.5) else 1.0
    cos, sin = math.cos(angle), math.sin(angle)
    theta = torch.tensor(
        [[cos * flip, -sin, tx], [sin * flip, cos, ty]], dtype=x.dtype, device=x.device
    ).unsqueeze(0).expand(s, -1, -1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    # pad with the normalised value of a black pixel
    fill = x.amin(dim=(0, 2, 3), keepdim=True)
    out = F.grid_sample(x - fill, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    return out + fill


class MRNetDataset(Dataset):
    """One item per exam: ``(uint8 volume (S, H, W), labels (n_findings,), case_id)``."""

    def __init__(
        self,
        root: str | Path,
        split: str,
        plane: str,
        case_ids: Sequence[str],
        labels: np.ndarray,
        image_size: int = 256,
    ) -> None:
        """Create the dataset.

        Args:
            root: MRNet root folder.
            split: Folder name, ``"train"`` or ``"valid"``.
            plane: ``"sagittal"``, ``"coronal"`` or ``"axial"``.
            case_ids: Case identifiers (zero-padded strings).
            labels: Array ``(len(case_ids), n_findings)``.
            image_size: In-plane size fed to the network.
        """
        self.dir = Path(root) / split / plane
        self.case_ids = list(case_ids)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.case_ids)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, str]:
        case = self.case_ids[idx]
        vol = to_uint8_volume(np.load(self.dir / f"{case}.npy"), self.image_size)
        return torch.from_numpy(vol), torch.from_numpy(self.labels[idx]), case


def collate_single(batch: list[tuple[torch.Tensor, torch.Tensor, str]]) -> tuple[torch.Tensor, torch.Tensor, str]:
    """Collate function for batch size 1 (exams have different slice counts)."""
    assert len(batch) == 1, "CartilAI trains one exam at a time (variable slice counts)"
    return batch[0]


def stratified_split(
    labels: np.ndarray, fraction: float, seed: int
) -> tuple[np.ndarray, np.ndarray]:
    """Split indices into train / internal-validation, stratified on the label pattern.

    Args:
        labels: ``(n, n_findings)`` binary labels.
        fraction: Fraction of cases to hold out.
        seed: Random seed.

    Returns:
        ``(train_idx, val_idx)`` sorted index arrays.
    """
    rng = np.random.default_rng(seed)
    keys = ["".join(str(int(v)) for v in row) for row in labels]
    train_idx: list[int] = []
    val_idx: list[int] = []
    for key in sorted(set(keys)):
        idx = np.array([i for i, k in enumerate(keys) if k == key])
        rng.shuffle(idx)
        n_val = int(round(len(idx) * fraction))
        if len(idx) >= 2:
            n_val = max(1, n_val)
        val_idx.extend(idx[:n_val].tolist())
        train_idx.extend(idx[n_val:].tolist())
    return np.array(sorted(train_idx)), np.array(sorted(val_idx))


In [ ]:
# ===== src/model.py (inlined) =====
"""CartilAI network: MRNet-style per-slice 2D CNN with max-pooling across slices.

Architecture (Bien et al., 2018, "Deep-learning-assisted diagnosis for knee MRI"),
with ResNet18 in place of AlexNet::

    volume (S, 3, H, W)
      -> ResNet18 feature extractor, applied to every slice   (S, 512, h, w)
      -> global average pool over each slice                    (S, 512)
      -> max-pool across slices                                 (512,)
      -> dropout + linear layer                                 (3,) logits
      -> sigmoid                                                one probability per finding
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torchvision


class CartilAINet(nn.Module):
    """Multi-label exam classifier for knee MRI volumes."""

    def __init__(self, num_findings: int = 3, pretrained: bool = True, dropout: float = 0.3) -> None:
        """Build the network.

        Args:
            num_findings: Number of outputs (one logit per finding).
            pretrained: Load ImageNet weights for the ResNet18 backbone. Falls back
                to random initialisation (with a warning) if they cannot be downloaded.
            dropout: Dropout probability before the classifier.
        """
        super().__init__()
        weights = None
        if pretrained:
            weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
        try:
            resnet = torchvision.models.resnet18(weights=weights)
        except Exception as exc:  # no internet: keep going with random weights
            print(f"[CartilAI] Could not load ImageNet weights ({exc}); using random initialisation.")
            resnet = torchvision.models.resnet18(weights=None)
        # Everything up to (and including) layer4; drop avgpool + fc.
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(512, num_findings)

    @property
    def gradcam_layer(self) -> nn.Module:
        """The last convolutional block, used as the Grad-CAM target layer."""
        return self.features[-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Score one exam.

        Args:
            x: Preprocessed volume ``(S, 3, H, W)``.

        Returns:
            Logits of shape ``(num_findings,)``.
        """
        fmap = self.features(x)                   # (S, 512, h, w)
        per_slice = self.pool(fmap).flatten(1)    # (S, 512)
        exam = per_slice.max(dim=0).values        # (512,)
        return self.classifier(self.dropout(exam))


def build_model(config: dict, pretrained: bool | None = None) -> CartilAINet:
    """Construct the model from the config dictionary.

    Args:
        config: Full CartilAI config.
        pretrained: Override ``config["model"]["pretrained"]``.

    Returns:
        A :class:`CartilAINet`.
    """
    mcfg = config["model"]
    return CartilAINet(
        num_findings=len(config["findings"]),
        pretrained=mcfg["pretrained"] if pretrained is None else pretrained,
        dropout=mcfg["dropout"],
    )


In [ ]:
# ===== src/evaluate.py (inlined) =====
"""Evaluation: AUC with bootstrap CIs, threshold selection, ROC curves and confusion matrices."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import torch
from sklearn.metrics import roc_auc_score, roc_curve



@torch.no_grad()
def predict_dataset(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    device: torch.device,
    amp: bool = False,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Run the model over a loader and collect probabilities.

    Args:
        model: Trained network.
        loader: Yields ``(volume, labels, case_id)`` with batch size 1.
        device: Device to run on.
        amp: Use autocast (CUDA only).

    Returns:
        ``(y_true, y_prob, case_ids)``; arrays have shape ``(n, n_findings)``.
    """
    model.eval()
    ys, ps, ids = [], [], []
    for vol, label, case in loader:
        x = preprocess(vol.to(device, non_blocking=True))
        with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
            logits = model(x)
        ps.append(torch.sigmoid(logits.float()).cpu().numpy())
        ys.append(label.numpy())
        ids.append(case)
    return np.stack(ys), np.stack(ps), ids


def safe_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """ROC AUC that returns NaN instead of raising when only one class is present."""
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))


def youden_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """Pick the probability threshold that maximises sensitivity + specificity - 1.

    Args:
        y_true: Binary labels.
        y_prob: Predicted probabilities.

    Returns:
        The threshold, or 0.5 if it cannot be computed.
    """
    if len(np.unique(y_true)) < 2:
        return 0.5
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    best = int(np.argmax(tpr - fpr))
    return float(np.clip(thr[best], 1e-6, 1 - 1e-6))


def bootstrap_auc_ci(
    y_true: np.ndarray, y_prob: np.ndarray, n_boot: int = 1000, seed: int = 0
) -> tuple[float, float]:
    """95% bootstrap confidence interval for the AUC.

    Args:
        y_true: Binary labels.
        y_prob: Predicted probabilities.
        n_boot: Number of bootstrap resamples.
        seed: Random seed.

    Returns:
        ``(low, high)``; NaNs if the AUC is undefined.
    """
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
    if not aucs:
        return float("nan"), float("nan")
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))


def compute_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    findings: Sequence[str],
    thresholds: Sequence[float],
    seed: int = 0,
) -> dict[str, dict[str, Any]]:
    """Compute per-finding metrics at the given decision thresholds.

    Args:
        y_true: ``(n, k)`` binary labels.
        y_prob: ``(n, k)`` probabilities.
        findings: Finding keys (length ``k``).
        thresholds: Decision threshold per finding.
        seed: Seed for the bootstrap.

    Returns:
        ``{finding: {auc, auc_ci_low, auc_ci_high, accuracy, sensitivity, specificity,
        threshold, tp, fp, tn, fn, n_positive, n_negative}}``.
    """
    out: dict[str, dict[str, Any]] = {}
    for k, finding in enumerate(findings):
        t, p = y_true[:, k], y_prob[:, k]
        pred = (p >= thresholds[k]).astype(int)
        tp = int(((pred == 1) & (t == 1)).sum())
        tn = int(((pred == 0) & (t == 0)).sum())
        fp = int(((pred == 1) & (t == 0)).sum())
        fn = int(((pred == 0) & (t == 1)).sum())
        lo, hi = bootstrap_auc_ci(t, p, seed=seed)
        out[finding] = {
            "auc": safe_auc(t, p),
            "auc_ci_low": lo,
            "auc_ci_high": hi,
            "accuracy": (tp + tn) / max(1, len(t)),
            "sensitivity": tp / max(1, tp + fn),
            "specificity": tn / max(1, tn + fp),
            "threshold": float(thresholds[k]),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "n_positive": int(t.sum()),
            "n_negative": int(len(t) - t.sum()),
        }
    return out


def plot_roc_curves(
    y_true: np.ndarray, y_prob: np.ndarray, findings: Sequence[str], names: dict[str, str], path: str | Path
) -> None:
    """Save one ROC curve per finding to a single PNG."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    colors = ["#0f766e", "#2563eb", "#b45309"]
    fig, ax = plt.subplots(figsize=(5.5, 5), dpi=130)
    for k, finding in enumerate(findings):
        t, p = y_true[:, k], y_prob[:, k]
        if len(np.unique(t)) < 2:
            continue
        fpr, tpr, _ = roc_curve(t, p)
        ax.plot(fpr, tpr, lw=2, color=colors[k % len(colors)],
                label=f"{names.get(finding, finding)} (AUC {safe_auc(t, p):.3f})")
    ax.plot([0, 1], [0, 1], ls="--", color="#94a3b8", lw=1)
    ax.set_xlabel("False positive rate (1 - specificity)")
    ax.set_ylabel("True positive rate (sensitivity)")
    ax.set_title("CartilAI — ROC curves (held-out MRNet validation set)")
    ax.legend(loc="lower right", frameon=False)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def plot_confusion_matrices(
    metrics: dict[str, dict[str, Any]], findings: Sequence[str], names: dict[str, str], path: str | Path
) -> None:
    """Save a row of 2x2 confusion matrices (one per finding) to a PNG."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, len(findings), figsize=(4.2 * len(findings), 3.9), dpi=130)
    axes = np.atleast_1d(axes)
    for ax, finding in zip(axes, findings):
        m = metrics[finding]
        cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
        ax.imshow(cm, cmap="Blues")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "#0f172a", fontsize=14)
        ax.set_xticks([0, 1], ["Pred. negative", "Pred. positive"])
        ax.set_yticks([0, 1], ["Negative", "Positive"])
        ax.set_title(f"{names.get(finding, finding)}\n(threshold {m['threshold']:.2f})")
    fig.suptitle("CartilAI — confusion matrices (held-out MRNet validation set)")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def plot_training_curves(history: list[dict[str, float]], path: str | Path) -> None:
    """Save training loss and internal-validation AUC per epoch to a PNG."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    epochs = [h["epoch"] for h in history]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.6), dpi=130)
    a1.plot(epochs, [h["train_loss"] for h in history], marker="o", label="train")
    a1.plot(epochs, [h["val_loss"] for h in history], marker="o", label="internal val")
    a1.set_xlabel("epoch"); a1.set_ylabel("BCE loss"); a1.legend(frameon=False); a1.grid(alpha=0.25)
    a2.plot(epochs, [h["val_mean_auc"] for h in history], marker="o", color="#0f766e")
    a2.set_xlabel("epoch"); a2.set_ylabel("mean AUC"); a2.set_title("Internal validation"); a2.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def save_json(obj: Any, path: str | Path) -> None:
    """Write ``obj`` as pretty JSON, converting NaN to null."""

    def clean(o: Any) -> Any:
        if isinstance(o, dict):
            return {k: clean(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)):
            return [clean(v) for v in o]
        if isinstance(o, (float, np.floating)):
            return None if not np.isfinite(o) else float(o)
        if isinstance(o, np.integer):
            return int(o)
        return o

    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(clean(obj), fh, indent=2)


In [ ]:
# ===== src/train.py (inlined) =====
"""Training pipeline: train on MRNet, select the best epoch, evaluate, export everything.

The same code runs locally (synthetic data, CPU) and in the Kaggle notebook
(real MRNet, GPU); ``scripts/build_notebook.py`` inlines it into the notebook.

Local usage::

    python -m src.train --synthetic --quick          # smoke test on generated data
    python -m src.train --data-root /path/to/MRNet-v1.0
"""

from __future__ import annotations

import argparse
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


WEIGHTS_NAME = "cartilai_resnet18.pt"
N_EXPORTED_SAMPLES = 6

# Tiny settings for fast smoke tests (CPU, synthetic data).
QUICK_OVERRIDES: dict[str, Any] = {
    "data": {"image_size": 96, "num_workers": 0, "internal_val_fraction": 0.25},
    "model": {"pretrained": False},
    "train": {"epochs": 2, "lr": 1.0e-4, "max_train_minutes": 10},
}


def _loader(ds: MRNetDataset, shuffle: bool, workers: int, seed: int) -> DataLoader:
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        ds, batch_size=1, shuffle=shuffle, num_workers=workers, collate_fn=collate_single,
        pin_memory=torch.cuda.is_available(), generator=g, persistent_workers=workers > 0,
    )


def train_model(
    config: dict[str, Any],
    root: Path,
    device: torch.device,
    log: Any = print,
) -> tuple[nn.Module, list[float], list[dict[str, float]], dict[str, Any]]:
    """Train CartilAINet on the MRNet ``train`` split with an internal validation split.

    The best epoch (highest mean internal-validation AUC) is kept. Decision
    thresholds are chosen on the internal split (Youden's J) so that the
    official ``valid`` split stays an untouched test set.

    Args:
        config: Full CartilAI config.
        root: MRNet root folder.
        device: Training device.
        log: Logging function.

    Returns:
        ``(model, thresholds, history, info)``.
    """
    seed = config["seed"]
    findings = config["findings"]
    dcfg, tcfg = config["data"], config["train"]
    set_seed(seed)

    case_ids, labels = read_labels(root, "train", findings)
    tr_idx, va_idx = stratified_split(labels, dcfg["internal_val_fraction"], seed)
    train_ds = MRNetDataset(root, "train", dcfg["plane"], [case_ids[i] for i in tr_idx], labels[tr_idx], dcfg["image_size"])
    val_ds = MRNetDataset(root, "train", dcfg["plane"], [case_ids[i] for i in va_idx], labels[va_idx], dcfg["image_size"])
    log(f"Train exams: {len(train_ds)} | internal validation exams: {len(val_ds)}")
    train_dl = _loader(train_ds, True, dcfg["num_workers"], seed)
    val_dl = _loader(val_ds, False, dcfg["num_workers"], seed)

    model = build_model(config).to(device)
    # Class-imbalance weighting: pos_weight = negatives / positives per finding.
    pos = labels[tr_idx].sum(0)
    neg = len(tr_idx) - pos
    pos_weight = torch.tensor(np.where(pos > 0, neg / np.maximum(pos, 1), 1.0), dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=tcfg["lr"], weight_decay=tcfg["weight_decay"])
    total_steps = max(1, tcfg["epochs"] * len(train_dl))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=tcfg["lr"], total_steps=total_steps, pct_start=0.1,
    )
    use_amp = bool(tcfg["amp"]) and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    aug_gen = torch.Generator().manual_seed(seed)

    best_auc, best_state, best_epoch = -1.0, None, 0
    history: list[dict[str, float]] = []
    start = time.time()
    budget_s = tcfg["max_train_minutes"] * 60
    stopped_early = False

    for epoch in range(1, tcfg["epochs"] + 1):
        model.train()
        t0, running, n = time.time(), 0.0, 0
        for vol, label, _ in train_dl:
            x = preprocess(vol.to(device, non_blocking=True))
            if tcfg["augment"]:
                x = augment(x, aug_gen, tcfg["rotation_degrees"], tcfg["translate_pixels"], tcfg["hflip"])
            y = label.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(x)
            loss = criterion(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()
            n += 1

        y_val, p_val, _ = predict_dataset(model, val_dl, device, use_amp)
        with torch.no_grad():
            val_loss = float(criterion(torch.logit(torch.tensor(p_val).clamp(1e-6, 1 - 1e-6)).to(device),
                                       torch.tensor(y_val).to(device)))
        aucs = [safe_auc(y_val[:, k], p_val[:, k]) for k in range(len(findings))]
        mean_auc = float(np.nanmean(aucs)) if not np.all(np.isnan(aucs)) else float(-val_loss)
        history.append({"epoch": epoch, "train_loss": running / max(1, n), "val_loss": val_loss,
                        "val_mean_auc": mean_auc, **{f"val_auc_{f}": a for f, a in zip(findings, aucs)}})
        log(f"Epoch {epoch:02d}/{tcfg['epochs']} | train loss {running / max(1, n):.4f} | "
            f"val loss {val_loss:.4f} | val AUC " + " ".join(f"{f}={a:.3f}" for f, a in zip(findings, aucs))
            + f" | {time.time() - t0:.0f}s")
        if mean_auc > best_auc:
            best_auc, best_epoch = mean_auc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epoch_time = time.time() - t0
        if time.time() - start + epoch_time > budget_s and epoch < tcfg["epochs"]:
            log(f"Stopping after epoch {epoch}: next epoch would exceed the {tcfg['max_train_minutes']} min budget.")
            stopped_early = True
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    y_val, p_val, _ = predict_dataset(model, val_dl, device, use_amp)
    thresholds = [youden_threshold(y_val[:, k], p_val[:, k]) for k in range(len(findings))]
    info = {
        "best_epoch": best_epoch,
        "best_internal_val_mean_auc": best_auc,
        "epochs_completed": len(history),
        "stopped_early_for_time": stopped_early,
        "train_minutes": (time.time() - start) / 60,
        "n_train": len(train_ds),
        "n_internal_val": len(val_ds),
    }
    log(f"Best epoch {best_epoch} (internal val mean AUC {best_auc:.3f}); thresholds "
        + ", ".join(f"{f}={t:.2f}" for f, t in zip(findings, thresholds)))
    return model, thresholds, history, info


def save_checkpoint(
    path: Path, model: nn.Module, config: dict[str, Any], thresholds: list[float], extra: dict[str, Any]
) -> None:
    """Save weights plus everything the app needs to use them (findings, thresholds, config)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "format": "cartilai-v1",
        "state_dict": {k: v.detach().cpu().float() if v.is_floating_point() else v.detach().cpu()
                       for k, v in model.state_dict().items()},
        "findings": list(config["findings"]),
        "thresholds": [float(t) for t in thresholds],
        "config": {"model": config["model"], "data": config["data"]},
        "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        **extra,
    }, path)


def export_samples(
    root: Path, config: dict[str, Any], case_ids: list[str], labels: np.ndarray,
    probs: np.ndarray, out_dir: Path, n: int = N_EXPORTED_SAMPLES,
) -> list[dict[str, Any]]:
    """Copy a few held-out exams (mixed findings) and a manifest for the app's demo gallery.

    Args:
        root: MRNet root.
        config: Full config.
        case_ids: Validation case ids.
        labels: Their labels ``(n, k)``.
        probs: Model probabilities ``(n, k)``.
        out_dir: Destination folder.
        n: Number of exams to export.

    Returns:
        The manifest entries written to ``samples.json``.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    findings, names = config["findings"], config["finding_names"]
    rng = np.random.default_rng(config["seed"])
    # Aim for variety: normal, abnormal-only, ACL tear, meniscal tear, both.
    patterns = [(0, 0, 0), (1, 0, 0), (1, 1, 0), (1, 0, 1), (1, 1, 1), (1, 1, 0)]
    chosen: list[int] = []
    for pat in patterns:
        idx = [i for i in range(len(case_ids)) if tuple(int(v) for v in labels[i]) == pat and i not in chosen]
        if idx:
            chosen.append(int(rng.choice(idx)))
    for i in rng.permutation(len(case_ids)):
        if len(chosen) >= n:
            break
        if int(i) not in chosen:
            chosen.append(int(i))
    manifest = []
    plane = config["data"]["plane"]
    for i in chosen[:n]:
        case = case_ids[i]
        fname = f"mrnet_{plane}_{case}.npy"
        shutil.copy(root / "valid" / plane / f"{case}.npy", out_dir / fname)
        truth = [names[f] for f, v in zip(findings, labels[i]) if v == 1]
        manifest.append({
            "file": fname,
            "title": f"MRNet exam {case} ({plane})",
            "source": "MRNet v1.0 validation set (held out from training)",
            "ground_truth": {f: int(v) for f, v in zip(findings, labels[i])},
            "description": "Reference labels: " + (", ".join(truth) if truth else "normal"),
            "model_probabilities": {f: round(float(p), 4) for f, p in zip(findings, probs[i])},
        })
    with open(out_dir / "samples.json", "w", encoding="utf-8") as fh:
        json.dump(manifest, fh, indent=2)
    return manifest


def run_pipeline(
    config: dict[str, Any],
    input_dir: str | Path,
    output_dir: str | Path,
    log: Any = print,
) -> dict[str, Any]:
    """End-to-end: find data, train, evaluate on the held-out split, save all outputs.

    Output layout (mirrors the project folders so it can be copied in directly)::

        <output_dir>/models/cartilai_resnet18.pt
        <output_dir>/results/{metrics.json, roc_curves.png, confusion_matrices.png,
                              training_curves.png, predictions.csv}
        <output_dir>/samples/mrnet/{*.npy, samples.json}

    Args:
        config: Full CartilAI config.
        input_dir: Folder containing an MRNet dataset (searched recursively).
        output_dir: Where to write the outputs.
        log: Logging function.

    Returns:
        The metrics dictionary saved to ``metrics.json``.
    """
    t_start = time.time()
    set_seed(config["seed"])
    output_dir = Path(output_dir)
    models_dir, results_dir, samples_dir = output_dir / "models", output_dir / "results", output_dir / "samples" / "mrnet"
    for d in (models_dir, results_dir, samples_dir):
        d.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

    root = find_mrnet_root(input_dir, config["data"]["plane"])
    log(f"MRNet root: {root}")
    findings, names = config["findings"], config["finding_names"]

    model, thresholds, history, info = train_model(config, root, device, log)

    # Held-out evaluation on the official MRNet validation split (never used for training or selection).
    test_ids, test_labels = read_labels(root, "valid", findings)
    test_ds = MRNetDataset(root, "valid", config["data"]["plane"], test_ids, test_labels, config["data"]["image_size"])
    test_dl = _loader(test_ds, False, config["data"]["num_workers"], config["seed"])
    y_true, y_prob, ids = predict_dataset(model, test_dl, device, bool(config["train"]["amp"]))
    per_finding = compute_metrics(y_true, y_prob, findings, thresholds, seed=config["seed"])

    metrics = {
        "project": "CartilAI",
        "evaluation_set": f"MRNet v1.0 validation split ({len(ids)} exams, {config['data']['plane']} plane), held out from training",
        "model": f"{config['model']['backbone']} per-slice CNN + max-pool across slices",
        "plane": config["data"]["plane"],
        "findings": {f: {"name": names[f], **per_finding[f]} for f in findings},
        "mean_auc": float(np.nanmean([per_finding[f]["auc"] for f in findings])),
        "thresholds_selected_on": "internal validation split (10% of MRNet train), Youden's J",
        "training": {**info, "epochs_planned": config["train"]["epochs"], "lr": config["train"]["lr"],
                     "seed": config["seed"], "device": str(device),
                     "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None},
        "history": history,
        "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    save_json(metrics, results_dir / "metrics.json")
    plot_roc_curves(y_true, y_prob, findings, names, results_dir / "roc_curves.png")
    plot_confusion_matrices(per_finding, findings, names, results_dir / "confusion_matrices.png")
    plot_training_curves(history, results_dir / "training_curves.png")
    with open(results_dir / "predictions.csv", "w", encoding="utf-8") as fh:
        fh.write("case_id," + ",".join(f"label_{f}" for f in findings) + "," + ",".join(f"prob_{f}" for f in findings) + "\n")
        for c, t, p in zip(ids, y_true, y_prob):
            fh.write(c + "," + ",".join(str(int(v)) for v in t) + "," + ",".join(f"{v:.5f}" for v in p) + "\n")

    save_checkpoint(models_dir / WEIGHTS_NAME, model, config, thresholds,
                    {"test_metrics": {f: {"auc": per_finding[f]["auc"]} for f in findings}, "training": info})
    export_samples(root, config, ids, y_true, y_prob, samples_dir)

    log("\n=== CartilAI held-out results ===")
    for f in findings:
        m = per_finding[f]
        log(f"{names[f]:<14} AUC {m['auc']:.3f} (95% CI {m['auc_ci_low']:.3f}-{m['auc_ci_high']:.3f}) | "
            f"sens {m['sensitivity']:.2f} | spec {m['specificity']:.2f} | acc {m['accuracy']:.2f}")
    log(f"Total time: {(time.time() - t_start) / 60:.1f} min. Outputs in {output_dir}")
    return metrics


## Run training and evaluation

In [ ]:
# ===== Train, evaluate and export =====
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
metrics = run_pipeline(CONFIG, INPUT_DIR, OUTPUT_DIR)

In [ ]:
# ===== Show the results =====
from IPython.display import Image as IPImage, display, Markdown

rows = ["| Finding | AUC (95% CI) | Sensitivity | Specificity | Accuracy |", "|---|---|---|---|---|"]
for key, m in metrics["findings"].items():
    rows.append(f"| {m['name']} | {m['auc']:.3f} ({m['auc_ci_low']:.3f}-{m['auc_ci_high']:.3f}) | "
                f"{m['sensitivity']:.1%} | {m['specificity']:.1%} | {m['accuracy']:.1%} |")
display(Markdown("### Held-out MRNet validation results\n" + "\n".join(rows)))
for name in ["roc_curves.png", "confusion_matrices.png", "training_curves.png"]:
    display(IPImage(filename=str(OUTPUT_DIR / "results" / name)))

In [ ]:
# ===== Package everything into one zip for download =====
zip_path = WORK_DIR / "cartilai_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, Path("cartilai_outputs") / path.relative_to(OUTPUT_DIR))
print(f"Created {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB) containing:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(WORK_DIR))
print("\nDONE. Download 'cartilai_outputs.zip' from the Output panel and follow KAGGLE_STEPS.md.")

## Next steps
1. In the right-hand **Output** panel (or the notebook's *Output* tab after a saved run), download **`cartilai_outputs.zip`**.
2. Put the zip file in the CartilAI project folder (next to `app.py`), without unzipping it.
3. Run the launch command (`python start.py`). It installs the files into `models/`, `results/` and `samples/mrnet/` and the app switches from demo mode to the trained model.

MRNet data are subject to the Stanford MRNet research use agreement: keep the exported sample scans private.